<a href="https://colab.research.google.com/github/ibarr123/BUS1182026/blob/main/PART_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PART 2:  Code Generation with ReACT Prompting. Tools used: Google Colab + Google Gemini API (google-genai Python SDK)

In [8]:
!pip -q install -U google-genai

from google.colab import userdata
from google import genai

api_key = userdata.get("OPENAI_API_KEY")  # your AIzaSy... key saved in Secrets
if not api_key:
    raise ValueError("Missing Colab Secret: OPENAI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready ✅")

Gemini client ready ✅


BELOW IS ReACT PROMPT USED:


In [10]:
import re
import io
import traceback
from contextlib import redirect_stdout, redirect_stderr

REACT_PROMPT = """
You are generating Python 3 code to run in Google Colab.

You MUST follow this iterative ReACT cycle with clear section headers:

REASON/PLAN:
- Plan steps (3–6 bullets)
- State assumptions
- Mention edge cases and basic error handling

ACT (GENERATE CODE):
- Output ONE code block only using this exact format:

<START_CODE>
# python code here
<END_CODE>

- Must run in Colab as-is
- Use Python standard library only
- Include basic error handling

RUN:
- I will run your code exactly as written.

OBSERVE:
- If there is an error, I will paste the traceback.

FIX:
- If you get OBSERVATION with errors, revise plan and output corrected code.

Task:
{task}
"""

def extract_code(text):
    match = re.search(r"<START_CODE>(.*?)<END_CODE>", text, re.DOTALL)
    return match.group(1).strip() if match else ""

def run_code(code):
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    try:
        with redirect_stdout(stdout_buffer), redirect_stderr(stderr_buffer):
            exec(code, globals(), globals())
        output = stdout_buffer.getvalue() + stderr_buffer.getvalue()
        return True, output if output else "(No output)"
    except Exception:
        error_trace = traceback.format_exc()
        return False, error_trace

def react_loop(task, max_iters=3):
    observation = ""
    base_prompt = REACT_PROMPT.format(task=task)

    for i in range(1, max_iters + 1):
        prompt = base_prompt if not observation else base_prompt + "\n\nOBSERVATION:\n" + observation

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        text = response.text or ""
        print("\n" + "="*80)
        print(f"ITERATION {i} — MODEL RESPONSE")
        print("="*80)
        print(text)

        code = extract_code(text)
        if not code:
            observation = "ERROR: No code block found between <START_CODE> and <END_CODE>."
            continue

        print("\n" + "-"*80)
        print(f"ITERATION {i} — EXECUTION OUTPUT")
        print("-"*80)

        success, output = run_code(code)
        print(output)

        if success:
            print("\n✅ SUCCESS: Code ran without errors.")
            return

        observation = output

    print("\n❌ Stopped after max iterations.")

BELOW IS ReACT Code Generation Execution:

In [12]:
task = """
Write Python code that:
1) Prompts the user to enter a comma-separated list of numbers.
2) Converts valid values to floats.
3) Ignores invalid entries but reports which were invalid.
4) Prints count, sum, mean, min, and max.
5) Handles empty input safely.
"""

react_loop(task)


ITERATION 1 — MODEL RESPONSE
REASON/PLAN:
1.  Prompt the user for a comma-separated list of numbers using `input()`.
2.  Handle empty input: If the stripped input string is empty, print a message and exit gracefully.
3.  Process the input string: Split the string by commas. For each potential number:
    a.  Strip leading/trailing whitespace.
    b.  Attempt to convert it to a float using a `try-except ValueError` block.
    c.  Store successfully converted floats in a `valid_numbers` list.
    d.  Store the original (stripped) string of failed conversions in an `invalid_entries` list.
4.  Check for valid numbers: If `valid_numbers` is empty after parsing, print a message indicating no valid numbers were found, report any invalid entries, and exit.
5.  Calculate and print statistics: If valid numbers exist, calculate count (`len`), sum (`sum`), mean (`sum/len`), minimum (`min`), and maximum (`max`). Print these results clearly.
6.  Report invalid entries: If the `invalid_entries` list